# Hybrid Search: Combining BM25 + Vector Similarity

This notebook demonstrates the `HybridSearchService` which combines lexical (BM25) and semantic (vector) search using Reciprocal Rank Fusion (RRF).

## Why Hybrid Search?

- **Vector search** is great for semantic similarity ("fast database" matches "quick data store")
- **BM25 text search** is great for exact keyword matches ("Redis" finds documents with "Redis")
- **Hybrid search** combines both for best of both worlds

## Features

- Configurable weights for vector vs text search
- Reciprocal Rank Fusion (RRF) for score combination
- Automatic deduplication of results
- Metadata filtering support

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from redis_openai_agents import HybridSearchService, SearchResult

# Configuration
REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")

print(f"Redis URL: {REDIS_URL}")

## 1. Initialize HybridSearchService

The service manages both a vector store and a full-text search index internally.

In [ ]:
# Create hybrid search service with default weights (70% vector, 30% text)
service = HybridSearchService(
    redis_url=REDIS_URL,
    index_name="docs_demo",
    default_vector_weight=0.7,
    default_text_weight=0.3,
)

print(f"Index name: {service.index_name}")
print(f"Default vector weight: {service.default_vector_weight}")
print(f"Default text weight: {service.default_text_weight}")

## 2. Index Sample Documents

Let's index some documents about databases and programming.

In [ ]:
# Clear any existing data
await service.delete_all()

# Sample documents
documents = [
    {
        "content": "Redis is an in-memory data structure store used as a database, cache, and message broker.",
        "metadata": {"category": "database", "difficulty": "beginner"}
    },
    {
        "content": "PostgreSQL is a powerful open-source relational database management system.",
        "metadata": {"category": "database", "difficulty": "intermediate"}
    },
    {
        "content": "Python is a high-level programming language known for its simplicity and readability.",
        "metadata": {"category": "programming", "difficulty": "beginner"}
    },
    {
        "content": "Vector databases store high-dimensional embeddings for similarity search applications.",
        "metadata": {"category": "database", "difficulty": "advanced"}
    },
    {
        "content": "Machine learning models can be deployed using Redis for real-time inference.",
        "metadata": {"category": "ml", "difficulty": "advanced"}
    },
    {
        "content": "Caching strategies improve application performance by storing frequently accessed data.",
        "metadata": {"category": "performance", "difficulty": "intermediate"}
    },
]

# Index documents
ids = await service.index_documents(documents)
print(f"Indexed {len(ids)} documents")
print(f"Document count: {await service.count()}")

## 3. Basic Hybrid Search

Search combines both semantic similarity and keyword matching.

In [ ]:
# Search for "Redis database"
results = await service.search(query="Redis database", k=3)

print("Query: 'Redis database'")
print("=" * 60)
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result.score:.4f}")
    print(f"   Content: {result.content[:80]}...")
    print(f"   Metadata: {result.metadata}")

## 4. Semantic Search (High Vector Weight)

With high vector weight, results favor semantic similarity over exact matches.

In [ ]:
# Search with semantic focus (90% vector, 10% text)
results = await service.search(
    query="data storage solutions",  # No exact match for these words
    k=3,
    vector_weight=0.9,
    text_weight=0.1,
)

print("Query: 'data storage solutions' (90% vector weight)")
print("=" * 60)
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result.score:.4f}")
    print(f"   Content: {result.content[:80]}...")

## 5. Keyword Search (High Text Weight)

With high text weight, results favor exact keyword matches.

In [ ]:
# Search with keyword focus (10% vector, 90% text)
results = await service.search(
    query="Redis",  # Exact keyword
    k=3,
    vector_weight=0.1,
    text_weight=0.9,
)

print("Query: 'Redis' (90% text weight)")
print("=" * 60)
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result.score:.4f}")
    print(f"   Content: {result.content[:80]}...")

## 6. Filtered Search

Combine hybrid search with metadata filtering.

In [ ]:
# Search only in "database" category
results = await service.search(
    query="fast data store",
    k=5,
    filter_expression={"category": "database"},
)

print("Query: 'fast data store' (filtered: category=database)")
print("=" * 60)
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result.score:.4f}")
    print(f"   Content: {result.content[:80]}...")
    print(f"   Category: {result.metadata.get('category', 'N/A')}")

## 7. Understanding RRF Scores

The Reciprocal Rank Fusion (RRF) score is calculated as:

```
score = sum(weight * (1 / (k + rank))) for each search method
```

Where `k=60` is a smoothing constant. Documents appearing in both vector and text results get higher combined scores.

In [ ]:
# Demonstrate deduplication - same query through both methods
results = await service.search(
    query="Redis",  # Should match in both vector and text
    k=5,
    vector_weight=0.5,
    text_weight=0.5,
)

print("Query: 'Redis' (equal weights - tests deduplication)")
print("=" * 60)
print(f"Total results: {len(results)} (deduplicated)")
for i, result in enumerate(results, 1):
    print(f"\n{i}. Score: {result.score:.4f}")
    print(f"   ID: {result.id[:30]}...")
    print(f"   Content: {result.content[:60]}...")

## 8. SearchResult Dataclass

Each result is a `SearchResult` with standardized fields.

In [ ]:
# Inspect SearchResult structure
results = await service.search(query="Python", k=1)

if results:
    result = results[0]
    print("SearchResult fields:")
    print(f"  - id: {result.id}")
    print(f"  - content: {result.content}")
    print(f"  - metadata: {result.metadata}")
    print(f"  - score: {result.score}")
    print(f"\nType: {type(result)}")

## 9. Use Case: RAG with Hybrid Search

Hybrid search is ideal for Retrieval-Augmented Generation (RAG) where you want to find relevant context for LLM prompts.

In [ ]:
async def get_context_for_query(query: str, k: int = 3) -> str:
    """Get relevant context for a user query using hybrid search."""
    results = await service.search(query=query, k=k)
    
    if not results:
        return "No relevant context found."
    
    context_parts = []
    for i, result in enumerate(results, 1):
        context_parts.append(f"{i}. {result.content}")
    
    return "\n".join(context_parts)

# Example: Get context for a question
user_question = "How can I improve my application's performance?"
context = await get_context_for_query(user_question)

print(f"User Question: {user_question}")
print("\nRetrieved Context:")
print("-" * 40)
print(context)

## Summary

The `HybridSearchService` provides:

1. **Combined search** - Best of BM25 and vector similarity
2. **Configurable weights** - Tune for your use case
3. **RRF scoring** - Intelligent score combination
4. **Deduplication** - No duplicate results
5. **Filtering** - Narrow results by metadata

Use hybrid search when:
- You need both exact matches AND semantic similarity
- Building RAG systems with diverse query types
- Users might search with keywords or natural language

In [ ]:
# Cleanup
await service.delete_all()
print("Cleaned up demo data.")